<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Compiler diagnostics: `warnings{...}` and `remarks{...}`

The compiler does more than translate your kernel to PTX -- it can also *inspect* it and tell you
when something looks wrong. Two opt-in options turn those inspections on:

- **`warnings{...}`** surfaces **legal-but-questionable** patterns: code that compiles and may run, but
  has a real chance of hanging, faulting, or behaving differently than you intended.
- **`remarks{...}`** surfaces **perf-only** findings: the program is functionally correct, just slower
  than it could be (for example, register spills into local memory).

Both are **non-fatal** -- compilation still succeeds and you still get a kernel. A third severity,
the **error**, is *always on* and *always fatal*: it fires on a proven defect, with or without any
option. This chapter shows one of each, end to end, and exactly how to switch the inspections on.

**You'll learn:** how to enable the compiler's diagnostic inspections with
`cute.compile(..., options="warnings{nvvm}")` / `"remarks{ptx}"` (or the matching `CUTE_DSL_COMPILER_OPT`
environment variable); the three diagnostic **severities** -- *remark* (perf), *warning*
(questionable), *error* (defect) -- and which are fatal; and how to catch a fatal diagnostic in Python with `CompilerDiagnosticError`.

**Runs on:** any CUDA GPU (the synchronization examples target sm_90+; the ptxas remark works on any
target). **Prereq:** Chapter on `cutlass.Array` and the memory spaces.

In [ ]:
import cutlass
import cutlass.cute as cute
from cutlass.experimental import primitives as prims 
from cutlass.base_dsl.compiler import CompilerDiagnosticError

## Levels and categories

You turn diagnostics on by passing an **options string** to `cute.compile`.
The string has two axes: the severity level to show and the diagnostic category to collect.
Warnings and remarks are opt-in and non-fatal. Errors reported by an enabled category are always
shown and fail compilation; there is no separate `errors{...}` option.

| Level | Enable with | Useful for | Fatal? |
|---|---|---|---|
| Info (remark) | `"remarks"` or `"remarks{<category>}"` | Performance-only findings, including synchronization opportunities and ptxas resource reports. | No |
| Warning | `"warnings"` or `"warnings{<category>}"` | Legal but questionable patterns that can hang, fault, or behave differently than intended. | No |
| Error | Enable the relevant category with `"warnings{<category>}"` or `"remarks{<category>}"`. | Proven defects reported by an enabled diagnostic category. | Yes |

| Category | Enable with | Source | Useful levels |
|---|---|---|---|
| `nvvm` | `"warnings{nvvm}"`, `"remarks{nvvm}"` | NVVM-level primitive protocol diagnostics for operations such as `mbarrier`, bulk copy, TMA multicast, and `tcgen05`. | Error, warning, info (remark) |
| `ptxas` (selector: `ptx`) | `"remarks{ptx}"` | ptxas resource diagnostics surfaced through the remark stream, including register spills and local-memory usage. | Info (remark) |

The exact same strings work as an environment variable, handy for turning diagnostics on without
editing code:

```bash
CUTE_DSL_COMPILER_OPT="warnings{nvvm}" python my_kernel.py
```

That is the whole mechanism. This chapter is about *how to switch the inspections on* -- not about
the specific findings -- so the three examples below show one of each severity end to end.

## 1. A warning -- legal but questionable

A *warning* fires on code that the compiler will happily build, but that carries a real risk at run
time. Here the kernel guards a uniform bulk copy with a full-mask `elect.sync` -- a correct
single-issuer idiom -- but the launch uses a **partial warp** of 4 threads. A full-mask elect in a
trailing partial warp can hang or fault, so the compiler flags it. It is *questionable*, not
*proven wrong*, so it is a **warning**: compilation still succeeds.

In [ ]:
@cute.kernel
def partial_warp_bulk_copy_kernel(gmem: cutlass.Array):
    # A single-issuer bulk copy: one elected lane drives the cp.async.bulk for the whole warp.
    mbar = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem, alignment=8)
    smem_dst = cutlass.Array(cutlass.Int32, 4, space=cutlass.AddressSpace.smem, alignment=16)

    if prims.elect_sync():  # full-mask elect: one lane of the (assumed full) warp
        prims.cp_async_bulk_shared_cluster_global(smem_dst, gmem, mbar, 16)


@cute.jit
def warning_host(gmem: cutlass.Array):
    # block=(4,1,1) is only a PARTIAL warp -> the elect.sync above becomes questionable.
    partial_warp_bulk_copy_kernel(gmem).launch(grid=(1, 1, 1), block=(4, 1, 1))


# A fake (compile-only) array + stream let us compile without launching on a real GPU.
gmem = cutlass.runtime.make_fake_compact_array(cutlass.Int32, (4,), assumed_align=16)

print(">>> compiling WITHOUT warnings{nvvm} (no diagnostic):")
cute.compile(warning_host, gmem)
print("    compiled, silent.\n")

print(">>> compiling WITH options='warnings{nvvm}':")
cute.compile(warning_host, gmem, options="warnings{nvvm}")
print("    compiled anyway -- the warning is non-fatal.")

# Expected output: the first compile is silent; the second prints a source-located
# warning before succeeding (compilation does NOT fail):
#
# warning[nvvm-diag:C13]: full-mask elect.sync guards ... in a partial-warp launch block
#   --> .../03_diagnostics...:NN:7
#      ...
#   suggestion: launch with a warp-multiple block size ...

## 2. An error -- a proven defect, always fatal

An *error* fires on a defect the compiler can prove, and it **always** fails compilation. There is no
"errors" flag: the compiler's checkers run whenever you ask for `warnings{nvvm}` (or `remarks{nvvm}`), and
once they run, any error they find is fatal regardless of which severity you asked to *see*.

This kernel initializes an `mbarrier` to expect **exactly one** arrival per phase, then lets *every*
thread arrive on it -- unguarded. That over-arrives the barrier and flips its phase unexpectedly: a
real bug, not a style nit. Because it is fatal, we wrap the compile in `try/except` and catch
`CompilerDiagnosticError`.

In [ ]:
@cute.kernel
def unguarded_arrive_kernel():
    mbar = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem, alignment=8)

    if prims.elect_sync():
        prims.mbarrier_init(mbar, 1)  # expect exactly ONE arrival per phase
    prims.fence_mbarrier_init()
    cute.arch.barrier()

    # BUG: no single-issuer guard, so all 32 threads arrive on a count=1 barrier.
    prims.mbarrier_arrive(mbar, count=1)
    prims.mbarrier_try_wait_parity(mbar, 0, time_limit=10_000_000)


@cute.jit
def error_host():
    unguarded_arrive_kernel().launch(grid=(1, 1, 1), block=(32, 1, 1))


print(">>> compiling WITH options='warnings{nvvm}' (runs the checkers):")
try:
    # warnings{nvvm} runs the checkers; any error they find is fatal no matter which severity you ask for.
    cute.compile(error_host, options="warnings{nvvm}")
    print("    unexpected: compile succeeded")
except CompilerDiagnosticError as exc:
    print("    compilation FAILED, as it should -- here is the diagnostic:\n")
    print(exc)

## 3. A remark -- correct, just slower

A *remark* is perf-only: the program is functionally correct, but the compiler noticed something that
costs performance. The classic example is a **register spill** -- when a kernel keeps too many values
live at once, `ptxas` runs out of registers and parks the overflow in local memory, adding traffic
and latency.

This kernel loads a 64-element window into registers and writes it back **reversed**, which keeps all
64 values live simultaneously. We compile it with a deliberately tight register budget
(`--maxrregcount=24`) to force the spill, then ask for the `ptx` remark category. Leave
`--remark-output` unset so the Python diagnostic renderer prints source frames. Because `ptxas`
reports spill totals at kernel granularity, the source frame marks the reported kernel / likely
pressure region rather than an exact spill instruction.

In [ ]:
_N = 64  # values per thread -- a full reversed window kept live forces register pressure


@cute.kernel
def reverse_store_kernel(src: cutlass.Array, dst: cutlass.Array):
    tid, _, _ = cute.arch.thread_idx()
    base = tid * cutlass.Int32(_N)
    window = src[base:_N]  # load the whole V-wide window into registers
    # Storing it reversed keeps every element of `window` live at once -> high register pressure.
    for i in cutlass.range_constexpr(_N):
        dst[base + cutlass.Int32(i)] = window[_N - 1 - i]


@cute.jit
def remark_host(src: cutlass.Array, dst: cutlass.Array):
    reverse_store_kernel(src, dst).launch(grid=(1, 1, 1), block=(32, 1, 1))


n = _N * 32
src = cutlass.runtime.make_fake_compact_array(cutlass.Int32, (n,), assumed_align=16)
dst = cutlass.runtime.make_fake_compact_array(cutlass.Int32, (n,), assumed_align=16)

print(">>> compiling WITH the ptx remark category + a tight register budget:")
# remarks{ptx} asks ptxas for perf remarks; --ptxas-options squeezes the register budget
# so the spill actually happens.
cute.compile(
    remark_host,
    src,
    dst,
    options=(
        "remarks{ptx} "
        "--ptxas-options '--maxrregcount=24 --override-directive-values'"
    ),
)
print("    compiled (the spill is perf-only, never fatal). See remark[ptxas] above.")

# Expected output: compilation succeeds and prints a ptxas spill remark with a source frame.

## Try it yourself

1. **Silence vs. surface.** Re-run section 1 with `options="remarks{nvvm}"` instead of `"warnings{nvvm}"`:
   the partial-warp *warning* no longer prints (it is gated to `warnings{nvvm}`), yet the compile still
   succeeds. Severity options control what you *see*, not what the compiler *checks*.
2. **The error is unconditional.** In section 2, drop the `options=` argument entirely. The
   over-arrive is a *defect*, so... it depends: with no diagnostic option the sync checker never runs, so the
   compile *succeeds silently*. Add `options="remarks{nvvm}"` and the same defect now fails the
   compile -- proof that asking for diagnostics (either severity) is what runs the checkers, after
   which the error is always fatal.
3. **Fix the bug.** Guard the arrive in section 2 with `if prims.elect_sync():` (or `if tid == 0:`,
   reading `tid` from `cute.arch.thread_idx()`) so a single thread arrives. Re-compile with
   `options="warnings{nvvm}"` -- the error is gone.
4. **Make the spill worse (or vanish).** In section 3, raise `_N` to 96 and watch the spill counts
   in the report climb; or drop the `--ptxas-options ...` part so ptxas keeps its full register budget,
   and the spill remark disappears -- there was nothing to spill.
5. **Use the environment variable.** Instead of `options="..."`, set
   `CUTE_DSL_COMPILER_OPT="warnings{nvvm}"` in your shell and run a plain `cute.compile(...)` with no
   options string -- same diagnostics, no code change.